# Advanced Problems with Solutions: Python 3.6 Dictionary Ordering

This notebook contains advanced practice problems focused on Python 3.6 dictionary ordering behavior.

Topics covered:

- Insertion order of regular dictionaries
- Updating existing keys without changing order
- Deleting and reinserting keys
- `dict.update()` ordering behavior
- `popitem()` behavior
- Moving keys to the end or front
- Differences between `dict` and `OrderedDict`
- Jupyter display caveats
- Building practical ordered dictionary utilities

Each problem includes a complete solution and verification tests.

## Problem 1: Track Dictionary Order Through Mutations

### Task

Write a function `trace_dict_order(operations)` that receives a list of operations and returns the order of dictionary keys after each operation.

Each operation is a tuple in one of these forms:

```python
('set', key, value)
('delete', key)
('update', key, value)
```

Rules:

1. Setting a new key appends it to the end.
2. Updating an existing key changes its value but keeps its position.
3. Deleting a key removes it from the order.
4. Setting a previously deleted key inserts it at the end again.

Return a list of key-order snapshots after each operation.

In [1]:
def trace_dict_order(operations):
    d = {}
    snapshots = []

    for operation in operations:
        action = operation[0]

        if action == 'set':
            _, key, value = operation
            d[key] = value

        elif action == 'update':
            _, key, value = operation
            if key not in d:
                raise KeyError(f'Cannot update missing key: {key!r}')
            d[key] = value

        elif action == 'delete':
            _, key = operation
            del d[key]

        else:
            raise ValueError(f'Unknown operation: {action!r}')

        snapshots.append(list(d.keys()))

    return snapshots


# Verification
operations = [
    ('set', 'a', 1),
    ('set', 'b', 2),
    ('set', 'x', 3),
    ('delete', 'b'),
    ('set', 'b', 4),
    ('update', 'x', 100)
]

result = trace_dict_order(operations)

expected = [
    ['a'],
    ['a', 'b'],
    ['a', 'b', 'x'],
    ['a', 'x'],
    ['a', 'x', 'b'],
    ['a', 'x', 'b']
]

assert result == expected
result

[['a'],
 ['a', 'b'],
 ['a', 'b', 'x'],
 ['a', 'x'],
 ['a', 'x', 'b'],
 ['a', 'x', 'b']]

## Problem 2: Stable Value Replacement

### Task

Write a function `replace_values_stably(d, replacements)`.

`d` is a dictionary.

`replacements` is another dictionary whose keys may or may not exist in `d`.

Rules:

1. If a replacement key already exists in `d`, update the value but preserve the original position.
2. If a replacement key does not exist in `d`, append it to the end.
3. Return a new dictionary.
4. Do not mutate the input dictionary.

This problem tests the fact that assigning a new value to an existing key does not move that key.

In [2]:
def replace_values_stably(d, replacements):
    result = dict(d)

    for key, value in replacements.items():
        result[key] = value

    return result


# Verification
original = {'a': 1, 'b': 2, 'c': 3}
replacements = {'b': 200, 'x': 99, 'a': 100}

result = replace_values_stably(original, replacements)

assert original == {'a': 1, 'b': 2, 'c': 3}
assert result == {'a': 100, 'b': 200, 'c': 3, 'x': 99}
assert list(result.keys()) == ['a', 'b', 'c', 'x']

result

{'a': 100, 'b': 200, 'c': 3, 'x': 99}

## Problem 3: Simulate `dict.update()` Ordering

### Task

Write a function `manual_update(left, right)` that behaves like:

```python
left.update(right)
```

but returns a new dictionary instead of mutating `left`.

Ordering rules:

1. Existing keys in `left` keep their original positions.
2. Values for shared keys are replaced by values from `right`.
3. New keys from `right` are appended to the end.
4. New keys from `right` appear in the order they appear in `right`.

In [3]:
def manual_update(left, right):
    result = dict(left)

    for key, value in right.items():
        result[key] = value

    return result


# Verification
d1 = {'a': 1, 'b': 200}
d2 = {'a': 100, 'd': 300, 'c': 400}

result = manual_update(d1, d2)

assert result == {'a': 100, 'b': 200, 'd': 300, 'c': 400}
assert list(result.keys()) == ['a', 'b', 'd', 'c']
assert d1 == {'a': 1, 'b': 200}

result

{'a': 100, 'b': 200, 'd': 300, 'c': 400}

## Problem 4: Move a Key to the End

### Task

Write a function `move_to_end_regular_dict(d, key)` that returns a new dictionary where `key` has been moved to the end.

Rules:

1. Use a regular dictionary, not `OrderedDict`.
2. Preserve all key-value pairs.
3. Do not mutate the original dictionary.
4. Raise `KeyError` if the key does not exist.

Hint:

Removing and reinserting a key places it at the end.

In [4]:
def move_to_end_regular_dict(d, key):
    if key not in d:
        raise KeyError(key)

    result = dict(d)
    result[key] = result.pop(key)
    return result


# Verification
d = {'a': 1, 'b': 2, 'c': 3}
result = move_to_end_regular_dict(d, 'a')

assert d == {'a': 1, 'b': 2, 'c': 3}
assert result == {'b': 2, 'c': 3, 'a': 1}
assert list(result.keys()) == ['b', 'c', 'a']

result

{'b': 2, 'c': 3, 'a': 1}

## Problem 5: Move a Key to the Front

### Task

Write a function `move_to_front_regular_dict(d, key)` that returns a new regular dictionary with `key` moved to the front.

Rules:

1. Do not use `OrderedDict`.
2. Do not mutate the original dictionary.
3. Preserve all key-value pairs.
4. Raise `KeyError` if the key does not exist.

Unlike moving a key to the end, moving a key to the front is less direct with a regular dictionary.

In [5]:
def move_to_front_regular_dict(d, key):
    if key not in d:
        raise KeyError(key)

    result = {key: d[key]}

    for current_key, value in d.items():
        if current_key != key:
            result[current_key] = value

    return result


# Verification
d = {'a': 1, 'b': 2, 'c': 3, 'x': 100, 'y': 200}
result = move_to_front_regular_dict(d, 'c')

assert d == {'a': 1, 'b': 2, 'c': 3, 'x': 100, 'y': 200}
assert result == {'c': 3, 'a': 1, 'b': 2, 'x': 100, 'y': 200}
assert list(result.keys()) == ['c', 'a', 'b', 'x', 'y']

result

{'c': 3, 'a': 1, 'b': 2, 'x': 100, 'y': 200}

## Problem 6: Pop First and Pop Last

### Task

Write two functions:

```python
pop_first(d)
pop_last(d)
```

Both functions should mutate the input dictionary and return the removed `(key, value)` pair.

Rules:

1. `pop_last(d)` should use `d.popitem()`.
2. `pop_first(d)` should remove the first inserted key.
3. Both should raise `KeyError` for an empty dictionary.

In Python 3.6, `popitem()` pops the right-most item from the dictionary.

In [6]:
def pop_last(d):
    if not d:
        raise KeyError('dictionary is empty')
    return d.popitem()


def pop_first(d):
    if not d:
        raise KeyError('dictionary is empty')
    first_key = next(iter(d))
    first_value = d.pop(first_key)
    return first_key, first_value


# Verification
d = {'a': 1, 'b': 2, 'c': 3}

first = pop_first(d)
last = pop_last(d)

assert first == ('a', 1)
assert last == ('c', 3)
assert d == {'b': 2}

first, last, d

(('a', 1), ('c', 3), {'b': 2})

## Problem 7: Implement a Lightweight LRU Cache with Regular Dict

### Task

Build a simple least-recently-used cache using a regular dictionary.

Write a class `SimpleLRUCache` with:

```python
cache = SimpleLRUCache(capacity=3)
cache.set(key, value)
cache.get(key)
cache.items()
```

Rules:

1. Recently used items should move to the end.
2. When capacity is exceeded, remove the first item.
3. `get` should return `None` for missing keys.
4. Use regular dictionaries only.

This problem demonstrates how dictionary ordering can support practical cache behavior, although `OrderedDict` is still often better suited for this exact use case.

In [7]:
class SimpleLRUCache:
    def __init__(self, capacity):
        if capacity <= 0:
            raise ValueError('capacity must be positive')
        self.capacity = capacity
        self._data = {}

    def _move_to_end(self, key):
        self._data[key] = self._data.pop(key)

    def _pop_first(self):
        first_key = next(iter(self._data))
        return first_key, self._data.pop(first_key)

    def set(self, key, value):
        if key in self._data:
            self._data[key] = value
            self._move_to_end(key)
        else:
            self._data[key] = value

        if len(self._data) > self.capacity:
            self._pop_first()

    def get(self, key):
        if key not in self._data:
            return None

        value = self._data[key]
        self._move_to_end(key)
        return value

    def items(self):
        return list(self._data.items())


# Verification
cache = SimpleLRUCache(capacity=3)
cache.set('a', 1)
cache.set('b', 2)
cache.set('c', 3)

assert cache.items() == [('a', 1), ('b', 2), ('c', 3)]

assert cache.get('a') == 1
assert cache.items() == [('b', 2), ('c', 3), ('a', 1)]

cache.set('d', 4)
assert cache.items() == [('c', 3), ('a', 1), ('d', 4)]
assert cache.get('b') is None

cache.items()

[('c', 3), ('a', 1), ('d', 4)]

## Problem 8: Compare Regular Dict with OrderedDict

### Task

Write a function `compare_move_operations()` that demonstrates the difference between regular `dict` and `OrderedDict` for moving items.

Return a dictionary with these keys:

```python
{
    'regular_move_to_end': ...,
    'ordered_move_to_end': ...,
    'ordered_move_to_front': ...,
    'ordered_pop_first': ...,
    'ordered_pop_last': ...
}
```

Use:

```python
{'a': 1, 'b': 2, 'c': 3}
```

as the starting data.

This problem highlights methods available on `OrderedDict` that regular dictionaries do not directly provide.

In [8]:
from collections import OrderedDict


def compare_move_operations():
    regular = {'a': 1, 'b': 2, 'c': 3}
    regular['a'] = regular.pop('a')

    ordered_end = OrderedDict([('a', 1), ('b', 2), ('c', 3)])
    ordered_end.move_to_end('a')

    ordered_front = OrderedDict([('a', 1), ('b', 2), ('c', 3)])
    ordered_front.move_to_end('c', last=False)

    ordered_pop_first = OrderedDict([('a', 1), ('b', 2), ('c', 3)])
    first = ordered_pop_first.popitem(last=False)

    ordered_pop_last = OrderedDict([('a', 1), ('b', 2), ('c', 3)])
    last = ordered_pop_last.popitem(last=True)

    return {
        'regular_move_to_end': list(regular.items()),
        'ordered_move_to_end': list(ordered_end.items()),
        'ordered_move_to_front': list(ordered_front.items()),
        'ordered_pop_first': first,
        'ordered_pop_last': last
    }


# Verification
result = compare_move_operations()

assert result['regular_move_to_end'] == [('b', 2), ('c', 3), ('a', 1)]
assert result['ordered_move_to_end'] == [('b', 2), ('c', 3), ('a', 1)]
assert result['ordered_move_to_front'] == [('c', 3), ('a', 1), ('b', 2)]
assert result['ordered_pop_first'] == ('a', 1)
assert result['ordered_pop_last'] == ('c', 3)

result

{'regular_move_to_end': [('b', 2), ('c', 3), ('a', 1)],
 'ordered_move_to_end': [('b', 2), ('c', 3), ('a', 1)],
 'ordered_move_to_front': [('c', 3), ('a', 1), ('b', 2)],
 'ordered_pop_first': ('a', 1),
 'ordered_pop_last': ('c', 3)}

## Problem 9: Detect Display Order Problems

### Task

In some notebook or pretty-printing contexts, the displayed dictionary representation may not match insertion order.

Write a function `order_report(d)` that returns a dictionary with:

```python
{
    'keys_order': ...,
    'items_order': ...,
    'safe_display': ...
}
```

Rules:

1. `keys_order` should be `list(d.keys())`.
2. `items_order` should be `list(d.items())`.
3. `safe_display` should be a string built manually from `d.items()`.

The goal is to create a reliable order report that does not depend on how an environment pretty-prints dictionaries.

In [9]:
def order_report(d):
    safe_display = '{' + ', '.join(
        f'{key!r}: {value!r}'
        for key, value in d.items()
    ) + '}'

    return {
        'keys_order': list(d.keys()),
        'items_order': list(d.items()),
        'safe_display': safe_display
    }


# Verification
d = {'x': 1, 'a': 2}
result = order_report(d)

assert result['keys_order'] == ['x', 'a']
assert result['items_order'] == [('x', 1), ('a', 2)]
assert result['safe_display'] == "{'x': 1, 'a': 2}"

result

{'keys_order': ['x', 'a'],
 'items_order': [('x', 1), ('a', 2)],
 'safe_display': "{'x': 1, 'a': 2}"}

## Problem 10: Ordered Audit Log Compressor

### Task

You receive an audit log as a list of events:

```python
events = [
    ('login', 'alice', '09:00'),
    ('login', 'bob', '09:05'),
    ('logout', 'alice', '09:30'),
    ('login', 'alice', '10:00')
]
```

Write a function `latest_event_by_user(events)`.

Return a dictionary mapping each user to their latest event.

Rules:

1. User order should reflect first appearance in the log.
2. Later events for the same user should update the value without changing the user's position.
3. Each value should be a dictionary with `action` and `time`.

Expected key order for the example:

```python
['alice', 'bob']
```

In [10]:
def latest_event_by_user(events):
    result = {}

    for action, user, time in events:
        result[user] = {
            'action': action,
            'time': time
        }

    return result


# Verification
events = [
    ('login', 'alice', '09:00'),
    ('login', 'bob', '09:05'),
    ('logout', 'alice', '09:30'),
    ('login', 'alice', '10:00')
]

result = latest_event_by_user(events)

assert list(result.keys()) == ['alice', 'bob']
assert result['alice'] == {'action': 'login', 'time': '10:00'}
assert result['bob'] == {'action': 'login', 'time': '09:05'}

result

{'alice': {'action': 'login', 'time': '10:00'},
 'bob': {'action': 'login', 'time': '09:05'}}

## Problem 11: Ordered Diff Between Two Dictionaries

### Task

Write a function `ordered_dict_diff(old, new)`.

Return a dictionary with three ordered sections:

```python
{
    'added': ..., 
    'removed': ..., 
    'changed': ...
}
```

Rules:

1. `added` should contain keys that appear only in `new`, ordered as they appear in `new`.
2. `removed` should contain keys that appear only in `old`, ordered as they appear in `old`.
3. `changed` should contain keys that appear in both but have different values, ordered as they appear in `old`.
4. Each changed value should be a tuple `(old_value, new_value)`.

In [11]:
def ordered_dict_diff(old, new):
    added = {}
    removed = {}
    changed = {}

    for key, value in new.items():
        if key not in old:
            added[key] = value

    for key, value in old.items():
        if key not in new:
            removed[key] = value

    for key, old_value in old.items():
        if key in new and new[key] != old_value:
            changed[key] = (old_value, new[key])

    return {
        'added': added,
        'removed': removed,
        'changed': changed
    }


# Verification
old = {'a': 1, 'b': 2, 'c': 3, 'x': 99}
new = {'b': 20, 'a': 1, 'd': 4, 'e': 5}

result = ordered_dict_diff(old, new)

assert list(result['added'].keys()) == ['d', 'e']
assert list(result['removed'].keys()) == ['c', 'x']
assert list(result['changed'].keys()) == ['b']
assert result['changed']['b'] == (2, 20)

result

{'added': {'d': 4, 'e': 5},
 'removed': {'c': 3, 'x': 99},
 'changed': {'b': (2, 20)}}

## Problem 12: Rebuild a Dictionary from Ordered Commands

### Task

Write a function `rebuild_from_commands(commands)`.

The function receives a list of command dictionaries.

Supported commands:

```python
{'op': 'set', 'key': 'a', 'value': 1}
{'op': 'delete', 'key': 'a'}
{'op': 'move_to_end', 'key': 'a'}
{'op': 'move_to_front', 'key': 'a'}
{'op': 'pop_first'}
{'op': 'pop_last'}
```

Return the final dictionary.

Rules:

1. Use regular dictionaries only.
2. Missing keys should raise `KeyError`.
3. Unsupported operations should raise `ValueError`.
4. The final dictionary order must be deterministic.

In [12]:
def rebuild_from_commands(commands):
    d = {}

    for command in commands:
        op = command['op']

        if op == 'set':
            d[command['key']] = command['value']

        elif op == 'delete':
            del d[command['key']]

        elif op == 'move_to_end':
            key = command['key']
            if key not in d:
                raise KeyError(key)
            d[key] = d.pop(key)

        elif op == 'move_to_front':
            key = command['key']
            if key not in d:
                raise KeyError(key)
            value = d[key]
            d = {key: value, **{k: v for k, v in d.items() if k != key}}

        elif op == 'pop_first':
            if not d:
                raise KeyError('dictionary is empty')
            key = next(iter(d))
            d.pop(key)

        elif op == 'pop_last':
            if not d:
                raise KeyError('dictionary is empty')
            d.popitem()

        else:
            raise ValueError(f'Unsupported operation: {op!r}')

    return d


# Verification
commands = [
    {'op': 'set', 'key': 'a', 'value': 1},
    {'op': 'set', 'key': 'b', 'value': 2},
    {'op': 'set', 'key': 'c', 'value': 3},
    {'op': 'move_to_end', 'key': 'a'},
    {'op': 'set', 'key': 'x', 'value': 100},
    {'op': 'move_to_front', 'key': 'c'},
    {'op': 'pop_last'}
]

result = rebuild_from_commands(commands)

assert list(result.keys()) == ['c', 'b', 'a']
assert result == {'c': 3, 'b': 2, 'a': 1}

result

{'c': 3, 'b': 2, 'a': 1}

# Summary

In this notebook, you practiced advanced uses of Python dictionary ordering:

- Assigning a new key appends it to the end.
- Updating an existing key does not move it.
- Deleting and reinserting a key moves it to the end.
- `dict.update()` preserves existing key positions and appends new keys from the merged dictionary.
- `popitem()` removes the last item.
- The first item can be removed with `next(iter(d))` followed by `pop`.
- `OrderedDict` still provides useful methods such as `move_to_end(key, last=False)` and `popitem(last=False)`.
- For reliable order inspection, prefer `list(d.keys())`, `list(d.items())`, or explicit manual display.

These techniques are useful when building deterministic APIs, serializers, cache systems, audit-log processors, and configuration tools.